Karen Mayerli Tahuite Ortiz  
**Carnet:** 202103058  

Burden, R. L., & Faires, J. D. (2011). *Análisis numérico* (10a ed.). Capítulo 3. Ejercicios 3.4 .

---

# Ejercicio 8

Un lanzador de béisbol lanza una bola rápida desde el montículo del lanzador hacia el receptor. A pesar de que la distancia desde el montículo hasta el plato de home es 60 pies y 6 pulgadas, normalmente, la bola viaja aproximadamente 55 pies y 5 pulgadas. Suponga que la velocidad inicial de la bola es 95 millas por hora y que la velocidad final en el plato de home es 92 millas por hora. Construya un polinomio de interpolación de Hermite para los datos:

| Tiempo t (s) | 0 | 0.4 |
|-------------|---|-----|
| Distancia d (ft) | 0 | 55.5 |
| Velocidad (mph) | 95 | 92 |

a) Use la derivada del polinomio de Hermite para calcular la velocidad de la bola en millas por hora en \( t = 0.2 \) segundos.  

b) ¿La velocidad máxima de la pelota se presenta en \( t = 0 \), o la derivada del polinomio de Hermite tiene un máximo que supera las 95 millas por hora? En ese caso, ¿esto parece razonable?  

**Sugerencia:** Convierta las millas por hora en pies por segundo para resolver el problema y después regrese a millas por hora para dar las respuestas.

**Burden, R. L., & Faires, J. D. — Análisis Numérico (10a ed.)**  
**Capítulo 3 — Página 104–105**  

---

# Algoritmo 3.3  
## Interpolación de Hermite

Para obtener los coeficientes del polinomio de interpolación de Hermite $ H(x) $ en los $ (n+1) $ nodos distintos $ x_0, x_1, \dots, x_n $:

---

### ENTRADA  
Números $ x_0, x_1, \dots, x_n $; valores $ f(x_0), \dots, f(x_n) $ y derivadas $ f'(x_0), \dots, f'(x_n) $.

---

### SALIDA  
Los coeficientes $ Q_{0,0}, Q_{1,1}, \dots, Q_{2n+1,2n+1} $ que determinan el polinomio:

$ H(x) = Q_{0,0} + Q_{1,1}(x - x_0) + Q_{2,2}(x - x_0)^2 $

$ + Q_{3,3}(x - x_0)^2(x - x_1) + \cdots $

$ + Q_{2n+1,2n+1}(x - x_0)^2(x - x_1)^2 \cdots (x - x_n) $

---

### Paso 1  
Para $ i = 0, 1, \dots, n $, realizar los pasos 2 y 3.

---

### Paso 2  
Definir:

$ z_{2i} = x_i, \quad z_{2i+1} = x_i $

$ Q_{2i,0} = f(x_i), \quad Q_{2i+1,0} = f(x_i) $

$ Q_{2i+1,1} = f'(x_i) $

---

### Paso 3  
Si $ i \neq 0 $, entonces:

$ Q_{2i,1} = \frac{Q_{2i,0} - Q_{2i-1,0}}{z_{2i} - z_{2i-1}} $

---

### Paso 4  
Para $ j = 2, 3, \dots, 2n+1 $:

Para $ i = j, j+1, \dots, 2n+1 $:

$ Q_{i,j} = \frac{Q_{i,j-1} - Q_{i-1,j-1}}{z_i - z_{i-j}} $

---

### Paso 5  
SALIDA:

$ (Q_{0,0}, Q_{1,1}, \dots, Q_{2n+1,2n+1}) $

FIN.

In [9]:
# Librerias 
import numpy as np
import pandas as pd

In [11]:
# Tiempo (s)
t = np.array([0.0, 0.4])

# Distancia d (ft)
d = np.array([0.0, 55.5])

# Velocidad (mph)
v_mph = np.array([95.0, 92.0])

# Conversión de millas por hora (mph) a oies por segundo (ft/s)
MPH_TO_FTPS = 1.46667
v = v_mph * MPH_TO_FTPS

# visualización de la tabla 
tabla = pd.DataFrame({
    "Tiempo (s)": t,
    "Distancia (ft)": d,
    "Velocidad (mph)": v_mph,
    "Velocidad (ft/s)": v
})

tabla

,Tiempo (s),Distancia (ft),Velocidad (mph),Velocidad (ft/s)
0,0.0,0.0,95.0,139.33365
1,0.4,55.5,92.0,134.93364


In [23]:
# Implementación del Algoritmo 3.3 (Burden)
def hermite_interpolation(x, fx, dfx):
    """
    Construye la tabla de diferencias divididas para la interpolación de Hermite.

    Parámetros
    ----------
    x   : Nodos de interpolación (x_0, x_1, ..., x_n)
    fx  : Valores de la función f(xi)
    dfx : Valores de la derivada f'(xi)

    Retorna
    -------
    z : Vector de nodos duplicados (z_0, z_1, ..., z_{2n+1})
    Q : Matriz de diferencias divididas
    """

    n = len(x) - 1  # Número de nodos originales

    # Tamaño total considerando duplicación de nodos
    m = 2 * (n + 1)

    z = np.zeros(m)       # Vector de nodos duplicados
    Q = np.zeros((m, m))  # Tabla de diferencias divididas

    # ====================================
    # Paso 1–3: Construcción inicial
    # ====================================
    for i in range(n + 1):

        # Duplicación de nodos
        z[2*i] = x[i]
        z[2*i + 1] = x[i]

        # Primera columna: valores de la función
        Q[2*i, 0] = fx[i]
        Q[2*i + 1, 0] = fx[i]

        # Segunda columna para nodos repetidos: derivadas
        Q[2*i + 1, 1] = dfx[i]

        # Segunda columna para nodos no repetidos
        if i != 0:
            Q[2*i, 1] = (Q[2*i, 0] - Q[2*i - 1, 0]) / (z[2*i] - z[2*i - 1])

    # ==================================================
    # Paso 4: Diferencias divididas de orden superior
    # ==================================================
    for j in range(2, m):  # Orden de la diferencia
        for i in range(j, m):  # Índice de la fila
            Q[i, j] = (Q[i, j-1] - Q[i-1, j-1]) / (z[i] - z[i-j])

    # Retorna el vector de nodos duplicados y la tabla completa de diferencias divididas
    return z, Q

In [ ]:
# visualización de la tabla de diferencias divididas
z, Q = hermite_interpolation(t, d, v)

pd.DataFrame(Q)

,0,1,2,3
0,0.0,0.00000,0.000000,0.000000
1,0.0,139.33365,0.000000,0.000000
2,55.5,138.75000,-1.459125,0.000000
3,55.5,134.93364,-9.540900,-20.204437


In [64]:
# visualización del polinomio de hermite:
import sympy as sp

def hermite_polynomial_forms(z, Q):
    """
    Construye el polinomio de Hermite en su forma estandar (expandida). 
    """
    x = sp.symbols('t')
    n = len(z)
    
    # Forma original 
    H_newton = sp.Float(Q[0,0])
    
    for i in range(1, n):
        term = sp.Float(Q[i,i])
        
        for j in range(i):
            term *= (x - sp.Float(z[j]))
        
        H_newton += term
    
    # Forma expandida
    H_expandido = sp.expand(H_newton)
        
    return H_newton, H_expandido 

H_newton, H_expandido = hermite_polynomial_forms(z, Q)

t = sp.symbols('t')
# Derivada
H_deriv = sp.diff(H_expandido, t)

# Mostrar 
print("\nForma estandar del polinomio de hermite. En este caso es de posición (distancia) en pies (ft):")
display(sp.Eq(sp.Symbol('H(t)'), H_expandido))

print("\nDerivada del polinomio (velocidad en ft/s):")
display(sp.Eq(sp.Symbol("H'(t)"), sp.simplify(H_deriv)))



Forma estandar del polinomio de hermite. En este caso es de posición (distancia) en pies (ft):


Eq(H(t), -20.2044375*t**3 + 6.62264999999998*t**2 + 139.33365*t)


Derivada del polinomio (velocidad en ft/s):


Eq(H'(t), -60.6133124999999*t**2 + 13.2453*t + 139.33365)

In [41]:
def hermite_eval(z, Q, x_val):
    """
    Evalúa el polinomio de interpolación de Hermite en un punto dado.

    Parámetros
    ----------
    Z, Q y x_val el cual es el punto en el cual se desea evaluar el polinomio

    Retorna
    -------
    result : Valor del polinomio de Hermite en x_val
    """

    # Inicializa el resultado con el primer coeficiente del polinomio
    result = Q[0, 0]

    # Inicializa el producto  (términos de la forma (x - z_i))
    product = 1.0

    # =========================
    # Evaluación  
    # =========================
    for i in range(1, len(z)):

        # Actualiza el producto  (x - z0)(x - z1)...(x - z_{i-1})
        product *= (x_val - z[i-1])

        # concatenar  términos 
        result += Q[i, i] * product

    # Retorna el valor evaluado del polinomio
    return result

In [65]:
# Aproximación de la derivada en x_val
def hermite_derivative(z, Q, x_val, h=1e-6):
    # Aproximación de la derivada usando diferencia central:
    # f'(x) ≈ [f(x + h) - f(x - h)] / (2h)
    return (hermite_eval(z, Q, x_val + h) - hermite_eval(z, Q, x_val - h)) / (2*h)

In [68]:
# Punto a valuar (tiempo en segundos)
t_eval = 0.2

# Cálculo de la velocidad en ft/s usando la derivada del polinomio de Hermite
v_ft_s = hermite_derivative(z, Q, t_eval)

# Conversión de ft/s a millas por hora (mph)
v_result_mph = v_ft_s / MPH_TO_FTPS

# resultado final
print(f"Velocidad en t = 0.2 s: {v_result_mph:.2f} mph")

Velocidad en t = 0.2 s: 95.15 mph


### a)
La velocidad de la bola de béisbol en millas por hora en $t = 0.2$ segundos es de:
$$
v(0.2) \approx 95.15 \ \text{mph}
$$

### b) ¿La velocidad máxima de la pelota se presenta en $t = 0$, o la derivada del polinomio de Hermite tiene un máximo que supera las $95$ millas por hora? En ese caso, ¿esto parece razonable?

En este caso, la derivada del polinomio de Hermite presenta un valor mayor a $95$ mph. Aunque en algunos sistemas físicos un objeto puede aumentar su velocidad debido a la acción de fuerzas externas (como ocurre en la caída libre, donde la gravedad acelera al objeto); En este caso la pelota de béisbol no experimenta ninguna fuerza que la acelere después de ser lanzada.

Dado que el movimiento de la pelota corresponde a un tiro parabólico donde la velocidad inicial es de $95$ mph en $t = 0$; conforme transcurre el tiempo, se espera que su velocidad disminuya debido a la resistencia del aire, la cual actúa en sentido opuesto al movimiento. Por lo tanto, un incremento en la velocidad después del lanzamiento **no resulta razonable**. El aumento es consecuencia de las oscilaciones del polinomio de interpolación así como del uso  reducido de datos.
